In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
# 데이터프레임을 불러오거나 생성합니다.
cluster_df = pd.read_csv('../P1_Data/02.Pretreatment/ClusteredData_gongju.csv',encoding='cp949')
cluster_df

In [ ]:
# 문자열 열을 제외
numeric_columns = cluster_df.select_dtypes(include=['number'])
# 클러스터 개수 범위 설정
n_clusters = range(1, 10)
inertia = []  # 관성 값 저장

# 각 클러스터 개수에 대한 K-Means 모델을 학습하고 관성 값을 저장
for k in n_clusters:
    kmeans = KMeans(n_clusters=k)
    kmeans.fit(numeric_columns)
    inertia.append(kmeans.inertia_)

# 관성 값 그래프 그리기
plt.plot(n_clusters, inertia, marker='o')
plt.xlabel('클러스터 개수')
plt.ylabel('클러스터 내의 관성')
plt.title('Elbow Method')
plt.show()

In [ ]:
cluster_df_stand = cluster_df.copy()
# 데이터를 표준화
scaler = StandardScaler()
cluster_df_stand.iloc[:,1:] = scaler.fit_transform(cluster_df_stand.iloc[:,1:] )
cluster_df_stand = cluster_df_stand.round(5)
cluster_scale_positive = cluster_df_stand.copy()

# 숫자형 컬럼만 선택
numeric_columns = cluster_scale_positive.select_dtypes(include=['number'])

# 각 숫자형 컬럼별 최솟값 출력
min_values = numeric_columns.min()
min_values

In [ ]:
cluster_scale_positive['주민등록_인구수'] = cluster_scale_positive['주민등록_인구수'] + 0.83369
cluster_scale_positive['주민등록_60세이상비율'] = cluster_scale_positive['주민등록_60세이상비율'] + 2.05291
cluster_scale_positive['유동인구_배율'] = cluster_scale_positive['유동인구_배율'] + 1.45796
cluster_scale_positive['유동인구_주중비율'] = cluster_scale_positive['유동인구_주중비율'] + 1.08498
cluster_scale_positive['카드_월사용액'] = cluster_scale_positive['카드_월사용액'] + 0.63551
cluster_scale_positive['카드_연증가율'] = cluster_scale_positive['카드_연증가율'] + 1.98887
cluster_scale_positive['카드_법인비율'] = cluster_scale_positive['카드_법인비율'] + 1.94216
cluster_scale_positive['카드_외부지역비율'] = cluster_scale_positive['카드_외부지역비율'] + 1.47419
cluster_scale_positive.to_csv('../P3_Result/01.CSV/5.유형화/공주시_평균포함_표준화.csv', index=False, encoding='cp949')
cluster_scale_positive

In [ ]:
cluster_df_stand = cluster_df_stand[cluster_df_stand['행정동명'] != '평균']
cluster_df_stand1 = cluster_df_stand[['행정동명','주민등록_인구수', '주민등록_60세이상비율','유동인구_배율', '유동인구_주중비율']]
scaled_data = cluster_df_stand1.set_index('행정동명')

k = 4  # 클러스터 수 설정

kmeans = KMeans(n_clusters=k, random_state=0)
kmeans.fit(scaled_data)

# 클러스터 결과를 확인
cluster_labels = kmeans.labels_
cluster_df_stand1['클러스터'] = cluster_labels
cluster_df_stand1

In [ ]:
# '행정동명'과 '클러스터'를 제외한 열 선택
columns_to_standardize = cluster_df_stand1.columns.difference(['행정동명', '클러스터'])

# 데이터프레임에서 선택한 열 추출
subset = cluster_df_stand1[columns_to_standardize]

# 표준화된 데이터를 데이터프레임으로 변환
scaled_df = pd.DataFrame(data=subset, columns=columns_to_standardize)

# '행정동명'과 '클러스터' 열을 가장 앞에 추가
scaled_df.insert(0, '클러스터', cluster_df_stand1['클러스터'])
scaled_df.insert(0, '행정동명', cluster_df_stand1['행정동명'])
scaled_df

In [ ]:
# 번호에 따라 행정동을 그룹화하고 해당 행정동을 합친 새로운 열 생성
grouped = scaled_df.groupby('클러스터')['행정동명'].apply(lambda x: ','.join(x)).reset_index()
# 열 이름 변경
grouped.columns = ['클러스터', '해당행정동']
scaled_df = pd.merge(scaled_df,grouped,on='클러스터',how='inner')
scaled_df.to_csv('../P3_Result/01.CSV/5.유형화/공주시_클러스터링결과_표준화(인구).csv',index=False,encoding='cp949')
scaled_df

In [ ]:
scaled_df_g = scaled_df.groupby(['클러스터','해당행정동'])[[ '주민등록_인구수','주민등록_60세이상비율','유동인구_배율', '유동인구_주중비율']].mean().round(1).reset_index()
scaled_df_g.min()

In [ ]:
scaled_df_g['주민등록_인구수'] = scaled_df_g['주민등록_인구수'] + 0.5
scaled_df_g['주민등록_60세이상비율'] = scaled_df_g['주민등록_60세이상비율'] + 1.8
scaled_df_g['유동인구_배율'] = scaled_df_g['유동인구_배율'] + 0.7
scaled_df_g['유동인구_주중비율'] = scaled_df_g['유동인구_주중비율'] + 0.6

scaled_df_g.to_csv('../P3_Result/01.CSV/5.유형화/공주시_클러스터링결과_유형별평균(인구).csv',index=False,encoding='cp949')
scaled_df_g

In [ ]:
cluster_hjd = cluster_df[['행정동명','주민등록_인구수','주민등록_60세이상비율','유동인구_배율', '유동인구_주중비율']]
hjd_cluster = scaled_df[['행정동명','클러스터']]
cluster_df_mg = pd.merge(cluster_hjd,hjd_cluster,on='행정동명')
cluster_df_mg.to_csv('../P3_Result/01.CSV/5.유형화/공주시_행정동별_클러스터유형(인구).csv',index=False,encoding='cp949')
cluster_df_mg

### 카드부분

In [ ]:
cluster_df_stand = cluster_df_stand[cluster_df_stand['행정동명'] != '평균']
cluster_df_stand2 = cluster_df_stand[['행정동명','카드_월사용액','카드_연증가율', '카드_법인비율', '카드_외부지역비율']]
scaled_data = cluster_df_stand2.set_index('행정동명')

k = 4  # 클러스터 수 설정

kmeans = KMeans(n_clusters=k, random_state=0)
kmeans.fit(scaled_data)

# 클러스터 결과를 확인
cluster_labels = kmeans.labels_
cluster_df_stand2['클러스터'] = cluster_labels
cluster_df_stand2

In [ ]:
# '행정동명'과 '클러스터'를 제외한 열 선택
columns_to_standardize = cluster_df_stand2.columns.difference(['행정동명', '클러스터'])

# 데이터프레임에서 선택한 열 추출
subset = cluster_df_stand2[columns_to_standardize]

# 표준화된 데이터를 데이터프레임으로 변환
scaled_df = pd.DataFrame(data=subset, columns=columns_to_standardize)

# '행정동명'과 '클러스터' 열을 가장 앞에 추가
scaled_df.insert(0, '클러스터', cluster_df_stand2['클러스터'])
scaled_df.insert(0, '행정동명', cluster_df_stand2['행정동명'])
scaled_df

In [ ]:
# 번호에 따라 행정동을 그룹화하고 해당 행정동을 합친 새로운 열 생성
grouped = scaled_df.groupby('클러스터')['행정동명'].apply(lambda x: ','.join(x)).reset_index()
# 열 이름 변경
grouped.columns = ['클러스터', '해당행정동']
scaled_df = pd.merge(scaled_df,grouped,on='클러스터',how='inner')
scaled_df.to_csv('../P3_Result/01.CSV/5.유형화/공주시_클러스터링결과_표준화(카드).csv',index=False,encoding='cp949')
scaled_df

In [ ]:
scaled_df_g = scaled_df.groupby(['클러스터','해당행정동'])[['카드_월사용액','카드_연증가율', '카드_법인비율', '카드_외부지역비율']].mean().round(1).reset_index()
scaled_df_g.min()

In [ ]:
scaled_df_g2 = scaled_df.groupby(['클러스터','해당행정동'])[['카드_월사용액','카드_연증가율', '카드_법인비율', '카드_외부지역비율']].mean().round(1).reset_index()
scaled_df_g2['카드_월사용액'] = scaled_df_g2['카드_월사용액'] + 0.6
scaled_df_g2['카드_연증가율'] = scaled_df_g2['카드_연증가율'] + 0.7
scaled_df_g2['카드_법인비율'] = scaled_df_g2['카드_법인비율'] + 1.8
scaled_df_g2['카드_외부지역비율'] = scaled_df_g2['카드_외부지역비율'] + 1.3
scaled_df_g2.to_csv('../P3_Result/01.CSV/5.유형화/공주시_클러스터링결과_유형별평균(카드).csv', index=False, encoding='cp949')
scaled_df_g2

In [ ]:
cluster_hjd2 = cluster_df[['행정동명','카드_월사용액','카드_연증가율', '카드_법인비율', '카드_외부지역비율']]
hjd_cluster = scaled_df[['행정동명','클러스터']]
cluster_df_mg = pd.merge(cluster_hjd2,hjd_cluster,on='행정동명')
cluster_df_mg.to_csv('../P3_Result/01.CSV/5.유형화/공주시_행정동별_클러스터유형(카드).csv',index=False,encoding='cp949')
cluster_df_mg